In [9]:
import polars as pl
import glob
from pathlib import Path

from clean_data import filter_states, remove_duplicates, remove_empty_values

In [10]:
COLS = [
    'ID', 'Posted', 'PickUp', 'CityOrigin', 'LatOrigin', 'LngOrigin',
    'CityDestination', 'LatDestination', 'LngDestination', 'Distance',
    'Source', 'Equip', 'StateOrigin', 'HubOrigin', 'StateDestination',
    'HubDestination', 'Broker_Shipper', 'RatePerMile', 'Weight', 'Size'
]

DIC = {
    'LatitudeOR':  'LatOrigin',
    'LongitudeOR': 'LngOrigin',
    'LatitudeDT':  'LatDestination',
    'LongitudeDT': 'LngDestination',
    'RateTotal':   'RatePerMile'
}

CARGAS_BASE = Path("../cargas drive")

EXCLUDE_FILES = {
    (CARGAS_BASE / "1/deltas/unprocessed/tpath_05-02-2024_00h17m15s.parquet").resolve(),
    (CARGAS_BASE / "10/deltas/processed/tpath_05-02-2024_00h17m15s.parquet").resolve(),
}

FOLDER_CONFIG = {
    1:  (['chr','dat','tql','tsmarter','freef','freightf','JonesL',
          'online_fs','prime','tpath','trulos','processed','unprocessed'],
         ['processed']),
    2:  (['chr','tsmarter','processed'],               ['processed']),
    3:  (['tsmarter','processed'],                     ['processed']),
    4:  (['tsmarter','processed'],                     ['processed']),
    5:  (['tsmarter','processed'],                     ['processed']),
    6:  (['tsmarter','processed'],                     ['processed']),
    7:  (['tsmarter','processed'],                     ['processed']),
    8:  (['tsmarter','processed'],                     ['processed']),
    9:  (['tsmarter','processed'],                     ['processed']),
    10: (['tsmarter','processed'],                     ['processed']),
    11: (['dat'],                                      ['unprocessed']),
    12: (['chr','freef','freightf','processed','tql'], ['processed','unprocessed']),
}

In [11]:
def normalize_file(df: pl.DataFrame) -> pl.DataFrame:
    """Normaliza columnas, tipos y RatePerMile de un parquet individual."""
    df = df.with_columns(pl.col('Distance').cast(pl.Float64, strict=False))

    if 'RateTotal' in df.columns:
        if 'RatePerMile' in df.columns:
            df = df.drop('RateTotal')
        else:
            df = df.with_columns(
                (pl.col('RateTotal').cast(pl.Float64, strict=False) / pl.col('Distance')).alias('RatePerMile')
            ).drop('RateTotal')

    rename_map = {k: v for k, v in DIC.items() if k in df.columns}
    if rename_map:
        df = df.rename(rename_map)

    # Normalizar timezones a UTC para evitar SchemaError en concat
    for col in ('Posted', 'PickUp'):
        if col in df.columns and df[col].dtype == pl.Datetime:
            tz = df[col].dtype.time_zone
            if tz:
                df = df.with_columns(pl.col(col).dt.convert_time_zone('UTC'))
            else:
                df = df.with_columns(pl.col(col).dt.replace_time_zone('UTC'))

    # Convertir columnas List[String] a String (algunos parquets guardan Equip como lista)
    for col in df.columns:
        if df[col].dtype == pl.List(pl.Utf8):
            df = df.with_columns(pl.col(col).list.join(', '))

    missing = [c for c in COLS if c not in df.columns]
    if missing:
        df = df.with_columns([pl.lit(None).alias(c) for c in missing])

    return df.select(COLS)


def fix_types(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col('ID').cast(pl.Utf8),
        pl.col('Weight').fill_null('').cast(pl.Utf8),
        pl.col('Size').fill_null('').cast(pl.Utf8),
        pl.col('Equip').fill_null('').cast(pl.Utf8),
        pl.col('LatOrigin').cast(pl.Float64, strict=False),
        pl.col('LatDestination').cast(pl.Float64, strict=False),
        pl.col('LngOrigin').cast(pl.Float64, strict=False),
        pl.col('LngDestination').cast(pl.Float64, strict=False),
        pl.col('Posted').cast(pl.Utf8).str.to_datetime(format=None, strict=False),
        pl.col('PickUp').cast(pl.Utf8).str.to_datetime(format=None, strict=False),
    ])


def process_lbd(folder_num: int, lbd: str, extra_subdirs: list) -> tuple:
    base = CARGAS_BASE / str(folder_num) / 'deltas' / lbd
    patterns = [str(base / '*.parquet')] + [str(base / sub / '*.parquet') for sub in extra_subdirs]

    files = list(set(f for pat in patterns for f in glob.glob(pat)))
    files = [f for f in files if Path(f).resolve() not in EXCLUDE_FILES]

    if not files:
        return None, None

    frames = []
    for file in files:
        try:
            df = pl.read_parquet(file)
            df = normalize_file(df)
            frames.append(df)
        except Exception as e:
            print(f'[ERROR] {file}: {e}')

    if not frames:
        return None, None

    df_full = pl.concat(frames, how='diagonal_relaxed')

    df_states = filter_states(df_full)
    print(f'  [{folder_num}/{lbd}] estados eliminados:    {df_full.height - df_states.height}')

    df_dedup = remove_duplicates(df_states)
    print(f'  [{folder_num}/{lbd}] duplicados eliminados: {df_states.height - df_dedup.height}')

    df_dedup = df_dedup.filter(
        ~((pl.col('RatePerMile') > 50) & (pl.col('Distance') > 50))
    )

    df_clean = remove_empty_values(df_dedup)
    print(f'  [{folder_num}/{lbd}] vacíos eliminados:     {df_dedup.height - df_clean.height}')

    return df_dedup, df_clean


def process_folders(folder_nums, results_gen: list, results_nan: list):
    for i in folder_nums:
        lbds, extra_subdirs = FOLDER_CONFIG[i]
        print(f'\n=== Carpeta {i} ===')
        for lbd in lbds:
            df_gen, df_nan = process_lbd(i, lbd, extra_subdirs)
            if df_gen is not None:
                results_gen.append(df_gen)
                results_nan.append(df_nan)

In [12]:
# Checkpoint 1 — carpetas 1 a 6
loadbs_gen_1 = []
loadbs_nan_1 = []

process_folders(range(1, 7), loadbs_gen_1, loadbs_nan_1)

print(f'\nTotal lbd procesados: {len(loadbs_gen_1)}')


=== Carpeta 1 ===
  [1/chr] estados eliminados:    9385
['487740643', '482408123', '484136809', '492151258', '464418771'] ...
609968
  [1/chr] duplicados eliminados: 53890863
  [1/chr] vacíos eliminados:     23
  [1/dat] estados eliminados:    179
['ZS0VGF8B', 'ZS0V7mjU', 'ZS0VT5CZ', 'ZS0Ve7J7', 'ZS0UyWTZ'] ...
3677933
  [1/dat] duplicados eliminados: 5484139
  [1/dat] vacíos eliminados:     0
  [1/tql] estados eliminados:    300
['9328045', '9119527', '9060066', '9719663', '7597018'] ...
1588169
  [1/tql] duplicados eliminados: 1237871
  [1/tql] vacíos eliminados:     3692
  [1/tsmarter] estados eliminados:    10857
['bd58ba0b-39af-4d26-9174-34ca33fd6c11', '590516e4-dcbc-4e91-ac64-6d815f53ac57', 'b23e4122-09ee-4901-aeb0-5025b886c283', '71bb5e92-d33a-4c03-8f5b-00f6e05a8a64', 'b9a6f064-b337-43fc-8201-820e952914c4'] ...
296965
  [1/tsmarter] duplicados eliminados: 540871
  [1/tsmarter] vacíos eliminados:     0
  [1/freef] estados eliminados:    0
['296339237', '296331808', '296333175', 

In [13]:
df_checkpoint1 = fix_types(pl.concat(loadbs_gen_1, how='diagonal_relaxed'))
df_checkpoint1.write_parquet('drivecheckpoint1.parquet')
print(f'Checkpoint 1 guardado: {df_checkpoint1.shape}')

Checkpoint 1 guardado: (7402572, 20)


In [14]:
# Checkpoint 2 — carpetas 7 a 12
loadbs_gen_2 = []
loadbs_nan_2 = []

process_folders(range(7, 13), loadbs_gen_2, loadbs_nan_2)

print(f'\nTotal lbd procesados: {len(loadbs_gen_2)}')


=== Carpeta 7 ===
  [7/tsmarter] estados eliminados:    299473
['2abb3a1f-af8b-4c86-bde9-ada23ab4ffc5', '8db72658-81b3-4487-a17f-3ea04124ca55', 'b5d37acb-8269-4d9f-8ee7-e83e241320c4', 'acf70588-08e4-4a5c-ae0b-3c80551ac2fb', '9571f0d1-0274-4656-902f-8282b1b4a71c'] ...
776677
  [7/tsmarter] duplicados eliminados: 19322495
  [7/tsmarter] vacíos eliminados:     0
  [7/processed] estados eliminados:    75
['zHZLnSjqeb8oLJAUKwozrQ==', 'LLAhZ1ex+DS3Ku0Vc', 'LLAh7cbW+ZS0JJvKG', 'LLAAd80M+ZS0EFFSR', 'LLAhEUry+ZS0JMVHj'] ...
709937
  [7/processed] duplicados eliminados: 461761
  [7/processed] vacíos eliminados:     1

=== Carpeta 8 ===
  [8/tsmarter] estados eliminados:    316046
['e4ba866d-885e-47b6-9581-00669095efea', '5a855f0f-521c-48e8-a777-c80714a9ae1f', '6b0d2434-f692-4e29-891d-953e7a99a203', 'c7a44171-ac39-4fdd-8b21-4a0276e74da6', '52b69714-4d75-44f2-920c-f8a78da9e271'] ...
1156780
  [8/tsmarter] duplicados eliminados: 20132207
  [8/tsmarter] vacíos eliminados:     0
  [8/processed] esta

In [15]:
df_checkpoint2 = fix_types(pl.concat(loadbs_gen_2, how='diagonal_relaxed'))
df_checkpoint2.write_parquet('drivecheckpoint2.parquet')
print(f'Checkpoint 2 guardado: {df_checkpoint2.shape}')

Checkpoint 2 guardado: (6775780, 20)


In [16]:
df_full = pl.concat([df_checkpoint1, df_checkpoint2], how='diagonal_relaxed')
df_full.write_parquet('drivecheckpoint_full.parquet')
print(f'Dataset completo guardado: {df_full.shape}')

Dataset completo guardado: (14178352, 20)
